# Atlas Slice Matching — Inference

Given a trained slice-encoder model, this notebook matches real-world coronal / sagittal / axial slice images to the closest slice in the atlas template.

## 0. Inputs

Fill in the paths below, then run the whole notebook (Run All).

In [ ]:
# ==== USER INPUT ====

# Directories containing the real-world images you want to match, one axis per folder.
SAGITTAL_DIR = "/kaggle/input/your-dataset/sagittal"
CORONAL_DIR  = "/kaggle/input/your-dataset/coronal"
AXIAL_DIR    = "/kaggle/input/your-dataset/axial"

# Path to the trained model checkpoint (.pt) produced by the training notebook,
# e.g. "model.pt" or "model_finetuned.pt". Upload it to this notebook and point here.
MODEL_PATH = "/kaggle/input/your-model/model.pt"

# Path to the atlas template NRRD used to build the slice bank the model matches against.
# Leave as-is if you're using the same atlas the model was trained on.
ATLAS_NRRD = "/kaggle/input/datasets/fatimanauman/atlastemplate/average_template_25.nrrd"

# ==== END USER INPUT ====


## 1. Setup

In [ ]:
!pip install -q timm pynrrd
import os, glob
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import timm
import nrrd
from PIL import Image, ImageOps
import torchvision.transforms as T
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)


## 2. Config

These must match the architecture/settings the model was trained with — leave them as-is unless you trained with different values.

In [ ]:
class CFG:
    ATLAS_NRRD = ATLAS_NRRD
    AXES = ["sagittal", "coronal", "axial"]
    AXIS_TO_DIM = {"sagittal": 2, "coronal": 0, "axial": 1}

    IMG_SIZE = 224
    BACKBONE = "convnext_base"
    EMB_DIM = 128

    MIN_CONTENT_STD = 15

    TEST_REALWORLD_DIRS = {
        "sagittal": SAGITTAL_DIR,
        "coronal": CORONAL_DIR,
        "axial": AXIAL_DIR,
    }
    REALWORLD_SAGITTAL_APPLY_ORIENT_FIX = False

    MODEL_CKPT = MODEL_PATH

    OUTPUT_DIR = "/kaggle/working"
    ATLAS_SLICE_DIR = os.path.join(OUTPUT_DIR, "atlas_slices")

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)


## 3. Atlas: Extract Canonical Slices from NRRD

In [ ]:
def orient_sagittal(img: Image.Image) -> Image.Image:
    img = ImageOps.mirror(img)
    img = img.rotate(90, expand=True)
    return img

def extract_atlas_slices(nrrd_path, out_dir, axis_to_dim):
    volume, header = nrrd.read(nrrd_path)
    volume = volume.astype(np.float32)
    vmin, vmax = volume.min(), volume.max()
    axis_sizes = {axis: volume.shape[dim] for axis, dim in axis_to_dim.items()}
    rows = []
    for axis, dim in axis_to_dim.items():
        axis_dir = os.path.join(out_dir, axis)
        os.makedirs(axis_dir, exist_ok=True)
        n = volume.shape[dim]
        for i in range(n):
            sl = np.take(volume, i, axis=dim)
            sl_norm = ((sl - vmin) / (vmax - vmin + 1e-8) * 255).astype(np.uint8)
            img = Image.fromarray(sl_norm).convert('L')
            if axis == 'sagittal':
                img = orient_sagittal(img)
            path = os.path.join(axis_dir, f"{i:04d}.png")
            img.save(path)
            rows.append({'path': path, 'axis': axis, 'slice_index': i})
    return pd.DataFrame(rows), axis_sizes

atlas_df, AXIS_SIZES = extract_atlas_slices(CFG.ATLAS_NRRD, CFG.ATLAS_SLICE_DIR, CFG.AXIS_TO_DIM)
print(atlas_df.groupby('axis').size())


### 3a. Filter out near-blank edge slices

In [ ]:
def slice_content_score(path):
    arr = np.array(Image.open(path))
    return arr.std()

atlas_df['content_score'] = atlas_df['path'].apply(slice_content_score)
before = len(atlas_df)
atlas_df = atlas_df[atlas_df['content_score'] >= CFG.MIN_CONTENT_STD].reset_index(drop=True)
print(f"dropped {before - len(atlas_df)} / {before} near-blank slices")
print(atlas_df.groupby('axis').size())

atlas_path_lookup = {(r['axis'], int(r['slice_index'])): r['path'] for _, r in atlas_df.iterrows()}


## 4. Model

In [ ]:
class SliceEncoder(nn.Module):
    def __init__(self, backbone_name, emb_dim):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.emb_head = nn.Linear(feat_dim, emb_dim)

    def forward(self, x):
        feats = self.backbone(x)
        return F.normalize(self.emb_head(feats), dim=-1)

model = SliceEncoder(CFG.BACKBONE, CFG.EMB_DIM).to(DEVICE)
model.load_state_dict(torch.load(CFG.MODEL_CKPT, map_location=DEVICE))
model.eval()
print(f"Loaded model from {CFG.MODEL_CKPT}")


## 5. Matching Utility

In [ ]:
BASE_TF = T.Compose([T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)), T.ToTensor()])

def load_gray(path):
    return Image.open(path).convert('L')

@torch.no_grad()
def build_atlas_bank(m, atlas_df, device=DEVICE):
    m = m.to(device).eval()
    embs, keys = [], []
    for _, row in atlas_df.iterrows():
        img = BASE_TF(load_gray(row['path']).convert('RGB')).unsqueeze(0).to(device)
        e = m(img)
        embs.append(e.cpu()); keys.append((row['axis'], int(row['slice_index'])))
    return torch.cat(embs, dim=0), keys

@torch.no_grad()
def match_slice(m, img_tensor, atlas_bank, atlas_keys, axis, device=DEVICE, topk=1):
    m.eval()
    e = m(img_tensor.unsqueeze(0).to(device)).cpu()
    sims = (e @ atlas_bank.T).squeeze(0)
    axis_mask = torch.tensor([0.0 if k[0] == axis else -1e9 for k in atlas_keys])
    sims = sims + axis_mask
    top = sims.topk(topk).indices
    return [(atlas_keys[i], sims[i].item()) for i in top]


## 6. Load Real-World Images

In [ ]:
def load_realworld_images(dirs):
    exts = ("*.png", "*.jpg", "*.jpeg", "*.jfif", "*.tif", "*.tiff", "*.bmp",
            "*.PNG", "*.JPG", "*.JPEG", "*.JFIF", "*.TIF", "*.TIFF", "*.BMP")
    records = []
    for axis, d in dirs.items():
        if not d or not os.path.isdir(d):
            print(f"[real-world] skipping '{axis}': directory not found -> {d}")
            continue
        paths = sorted(set(p for ext in exts for p in glob.glob(os.path.join(d, ext))))
        if not paths:
            print(f"[real-world] skipping '{axis}': no images found in {d}")
            continue
        for p in paths:
            records.append({"axis": axis, "path": p})
    return pd.DataFrame(records, columns=["axis", "path"])

realworld_df = load_realworld_images(CFG.TEST_REALWORLD_DIRS)
print(f"Found {len(realworld_df)} real-world image(s):")
if len(realworld_df):
    print(realworld_df.groupby('axis').size())


## 7. Run Inference

In [ ]:
banks = {}
for axis in realworld_df['axis'].unique():
    axis_atlas_df = atlas_df[atlas_df['axis'] == axis]
    banks[axis] = build_atlas_bank(model, axis_atlas_df)

results = []
for _, row in realworld_df.iterrows():
    axis, path = row['axis'], row['path']
    img = load_gray(path)
    if axis == 'sagittal' and CFG.REALWORLD_SAGITTAL_APPLY_ORIENT_FIX:
        img = orient_sagittal(img)
    tensor = BASE_TF(img.convert('RGB'))

    bank, keys = banks[axis]
    pred, score = match_slice(model, tensor, bank, keys, axis)[0]

    results.append(dict(
        path=path, axis=axis, query_tensor=tensor,
        pred_axis=pred[0], pred_slice_index=pred[1], score=score,
    ))

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'query_tensor'} for r in results])
results_df.to_csv(os.path.join(CFG.OUTPUT_DIR, "inference_results.csv"), index=False)
results_df


## 8. Visualize Results

In [ ]:
def show_tensor(ax, t, title=""):
    img = t.permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

if not results:
    print("No real-world images found -- check the directory paths in the input cell.")
else:
    n_rows = (len(results) + 1) // 2
    fig, axes = plt.subplots(n_rows, 4, figsize=(14, 4 * n_rows))
    if n_rows == 1:
        axes = axes[None, :]

    for i, r in enumerate(results):
        row_i, col_offset = i // 2, (i % 2) * 2
        fname = os.path.basename(r['path'])

        show_tensor(axes[row_i, col_offset], r['query_tensor'], f"QUERY ({r['axis']})\n{fname}")

        match_key = (r['pred_axis'], r['pred_slice_index'])
        match_tensor = BASE_TF(load_gray(atlas_path_lookup[match_key]).convert('RGB'))
        show_tensor(axes[row_i, col_offset + 1], match_tensor,
                    f"MATCH\n{match_key}\nsim={r['score']:.2f}")

    for i in range(len(results), n_rows * 2):
        row_i, col_offset = i // 2, (i % 2) * 2
        for c in range(2):
            axes[row_i, col_offset + c].axis('off')

    plt.tight_layout()
    plt.show()
